In [ ]:
import os
os.environ["HF_HOME"] = "/root/hf-cache"

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto"
)

print("Model     :", model_id)
print("GPU       :", torch.cuda.get_device_name(0))
print("Precision :", model.dtype)
print("VRAM      :", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/d46195ac87f837ad233d02b2f80f148bf7c005e0.2e01047a.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/d46195ac87f837ad233d02b2f80f148bf7c005e0.2e01047a.incomplete'.
Continuing without setting permissions.


tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/417d038a63fa3de29cfde265caedae14d1a58d92.6d1b6cf1.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/417d038a63fa3de29cfde265caedae14d1a58d92.6d1b6cf1.incomplete'.
Continuing without setting permissions.


vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/4783fe10ac3adce15ac8f358ef5462739852c569.6e02ce79.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/4783fe10ac3adce15ac8f358ef5462739852c569.6e02ce79.incomplete'.
Continuing without setting permissions.


... (10 outputs left out)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Could not set the permissions on the file '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/20a8a9156fc8c3f25295ca067f61fdf120d517c5.1cbba41b.incomplete'. Error: [Errno 1] Operation not permitted: '/workspace/.cache/huggingface/hub/models--Qwen--Qwen3-8B/blobs/20a8a9156fc8c3f25295ca067f61fdf120d517c5.1cbba41b.incomplete'.
Continuing without setting permissions.


Model     : Qwen/Qwen3-8B
GPU       : NVIDIA A100-SXM4-80GB
Precision : torch.bfloat16
VRAM      : 15.26 GB


BF16 모델 GPU에서 제거

In [3]:
import gc
import torch

del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

print("VRAM after cleanup:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

VRAM after cleanup: 0.0 GB


bitsandbytes 설치 - 4-bit 양자화

In [4]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 131.7 MB/s  0:00:00 eta 0:00:01


In [5]:
import bitsandbytes as bnb

print("bitsandbytes :", bnb.__version__)

bitsandbytes : 0.50.2


Qwen3-8B를 4-bit로 로딩

In [6]:
import os
os.environ["HF_HOME"] = "/root/hf-cache"

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

model_id = "Qwen/Qwen3-8B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto"
)

print("Model        :", model_id)
print("GPU          :", torch.cuda.get_device_name(0))
print("Quantization : 4-bit NF4")
print("Compute dtype:", torch.bfloat16)
print("VRAM         :",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model        : Qwen/Qwen3-8B
GPU          : NVIDIA A100-SXM4-80GB
Quantization : 4-bit NF4
Compute dtype: torch.bfloat16
VRAM         : 5.67 GB


4-bit 모델에 BF16 때와 동일한 질문으로 실습

In [7]:
import torch
import time

# BF16 실험과 완전히 동일한 질문
messages = [
    {
        "role": "user",
        "content": "생성형 AI와 기존 인공지능의 차이를 정확히 세 문장으로 설명해줘."
    }
]

# Thinking 비활성화
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

# 이번 추론의 Peak VRAM을 새로 측정
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )

torch.cuda.synchronize()
elapsed = time.time() - start

generated = outputs[0][inputs["input_ids"].shape[-1]:]

answer = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print("\n===== Qwen3-8B 4-bit Answer =====")
print(answer)

print("\n===== Performance =====")
print("Model            : Qwen/Qwen3-8B")
print("Quantization     : 4-bit NF4")
print("Compute dtype    : torch.bfloat16")
print("Generated tokens :", len(generated))
print("Generation time  :", round(elapsed, 3), "sec")
print("Tokens/sec       :", round(len(generated) / elapsed, 2))
print("VRAM allocated   :", round(torch.cuda.memory_allocated()/1024**3, 2), "GB")
print("VRAM peak        :", round(torch.cuda.max_memory_allocated()/1024**3, 2), "GB")


===== Qwen3-8B 4-bit Answer =====
생성형 AI는 새로운 내용을 생성하는 능력을 가진 인공지능으로, 텍스트, 이미지, 음성 등 다양한 형태의 데이터를 창작할 수 있습니다. 기존 인공지능은 주로 입력 데이터를 기반으로 특정 작업을 수행하는 분석형 기능을 중심으로 작동합니다. 생성형 AI는 기존 인공지능의 기능을 넘어, 창의적이고 유연한 새로운 정보를 만들어내는 역할을 합니다.

===== Performance =====
Model            : Qwen/Qwen3-8B
Quantization     : 4-bit NF4
Compute dtype    : torch.bfloat16
Generated tokens : 112
Generation time  : 5.735 sec
Tokens/sec       : 19.53
VRAM allocated   : 5.68 GB
VRAM peak        : 5.79 GB
